In [2]:
import pandas as pd
import numpy as np

In [3]:
#article_idlerin başındaki sıfırların silinmemesi için sondaki kod bloğunu ekledim.
df_articles = pd.read_csv(r"C:\Users\USER\Downloads\h-and-m-personalized-fashion-recommendations\articles.csv", dtype={'article_id': str})

#Satış verisi çok büyük olduğu için şimdilik 1 milyon satırı çektim
df_transactions = pd.read_csv(r"C:\Users\USER\Downloads\h-and-m-personalized-fashion-recommendations\transactions_train.csv", dtype={'article_id': str}, nrows=1000000)
df_articles.head()

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,0108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,0108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,0108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
3,0110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."
4,0110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."


In [4]:
print('Articles Veri Boyutu:', df_articles.shape)
print('Transactions Veri Boyutu:', df_transactions.shape)

Articles Veri Boyutu: (105542, 25)
Transactions Veri Boyutu: (1000000, 5)


In [5]:
df_transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 5 columns):
 #   Column            Non-Null Count    Dtype  
---  ------            --------------    -----  
 0   t_dat             1000000 non-null  object 
 1   customer_id       1000000 non-null  object 
 2   article_id        1000000 non-null  object 
 3   price             1000000 non-null  float64
 4   sales_channel_id  1000000 non-null  int64  
dtypes: float64(1), int64(1), object(3)
memory usage: 38.1+ MB


In [12]:
# t_dat sütununu gerçek tarih saat formatına çevirme
df_transactions['Tarih_Saat'] = pd.to_datetime(df_transactions['t_dat'])

#Sadece ay bilgisini ayrı bir sütun olarak çıkartma (sezonsal moda analizi için)
df_transactions['Ay'] = df_transactions['Tarih_Saat'].dt.month

#Sadece yıl bilgisini ayrı bür sütun olarak çıkarma
df_transactions['Yil'] = df_transactions['Tarih_Saat'].dt.year

# ilk 5 satırı görüntüleme
df_transactions[['t_dat', 'Tarih_Saat', 'Ay', 'Yil']].head()

,t_dat,Tarih_Saat,Ay,Yil
0,2018-09-20,2018-09-20,9,2018
1,2018-09-20,2018-09-20,9,2018
2,2018-09-20,2018-09-20,9,2018
3,2018-09-20,2018-09-20,9,2018
4,2018-09-20,2018-09-20,9,2018


In [17]:
#Aylara göre sezon belirleme (12,1,2: Kış | 3,4,5: İlkbahar | 6,7,8:Yaz | 9,10,11: Sonbahar)
def sezon_bul(ay):
    if ay in [12, 1, 2]:
        return 'Kis'
    elif ay in [3, 4, 5]:
        return 'İlkbahar'
    elif ay in [6, 7, 8]:
        return 'Yaz'
    else:
        return 'Sonbahar'

#Yeni sezon sütununu oluşturma
df_transactions['Sezon'] = df_transactions['Ay'].apply(sezon_bul)

#head yerine sample kullanarak veriden rastgele 10 satır çekme
df_transactions[['t_dat', 'Ay', 'Sezon']].sample(10)

,t_dat,Ay,Sezon
382253,2018-09-28,9,Sonbahar
517321,2018-09-29,9,Sonbahar
731012,2018-10-04,10,Sonbahar
608506,2018-10-01,10,Sonbahar
228773,2018-09-25,9,Sonbahar
42954,2018-09-20,9,Sonbahar
104752,2018-09-22,9,Sonbahar
738645,2018-10-04,10,Sonbahar
291133,2018-09-26,9,Sonbahar
335151,2018-09-27,9,Sonbahar


In [18]:
# Her sezondan toplam kaçar tane veri olduğunu sayma
df_transactions['Sezon'].value_counts()

Sezon
Sonbahar    1000000
Name: count, dtype: int64

In [20]:
#Satış verisi ile Ürün verisini 'article_id' ortak sütunu üzerinden birleştirme (Left Join)
print("Veriler birleştiriliyor...")
df_birlestirilmis = pd.merge(df_transactions, df_articles, on='article_id', how='left')

#Gereksiz ve kalabalık sütunları çıkarma ve sade bir analiz tablosu oluşturma
kullanilacak_sutunlar = [
    't_dat', 'article_id', 'price', 'sales_channel_id', 'Tarih_Saat', 'Ay', 'Yil', 'Sezon',
    'product_group_name', 'graphical_appearance_name', 'colour_group_name', 'garment_group_name' 
]

df_proje = df_birlestirilmis[kullanilacak_sutunlar]

#Birleştirilmiş tabloyu kontrol etme
print("\nBirleştirilmiş Yeni Tablonun Boyutu:", df_proje.shape)
df_proje.head()

Veriler birleştiriliyor...

Birleştirilmiş Yeni Tablonun Boyutu: (1000000, 12)


,t_dat,article_id,price,sales_channel_id,Tarih_Saat,Ay,Yil,Sezon,product_group_name,graphical_appearance_name,colour_group_name,garment_group_name
0,2018-09-20,0663713001,0.050831,2,2018-09-20,9,2018,Sonbahar,Underwear,Solid,Black,"Under-, Nightwear"
1,2018-09-20,0541518023,0.030492,2,2018-09-20,9,2018,Sonbahar,Underwear,Solid,Light Pink,"Under-, Nightwear"
2,2018-09-20,0505221004,0.015237,2,2018-09-20,9,2018,Sonbahar,Garment Upper body,Melange,Pink,Knitwear
3,2018-09-20,0685687003,0.016932,2,2018-09-20,9,2018,Sonbahar,Garment Upper body,Melange,Pink,Special Offers
4,2018-09-20,0685687004,0.016932,2,2018-09-20,9,2018,Sonbahar,Garment Upper body,Melange,Dark Green,Special Offers


In [21]:
#Her bir satırdaki eksik veri (NaN) sayısını bulma
print("--- Sütunlardakı Eksik Veri Sayıları ---")
print(df_proje.isnull().sum())

#Eksik veri içeren satırları temizleme (dropna)
df_proje_temiz = df_proje.dropna().copy()

#Temizleme işlemlerinden sonra verinin yeni boyutu
print("\n--- Temizleme Sonrası Veri Boyutu ---")
print(df_proje_temiz.shape)

--- Sütunlardakı Eksik Veri Sayıları ---
t_dat                        0
article_id                   0
price                        0
sales_channel_id             0
Tarih_Saat                   0
Ay                           0
Yil                          0
Sezon                        0
product_group_name           0
graphical_appearance_name    0
colour_group_name            0
garment_group_name           0
dtype: int64

--- Temizleme Sonrası Veri Boyutu ---
(1000000, 12)
